# AgentCore Runtime의 Middleware 지원

이 Notebook에서는 Amazon Bedrock AgentCore Runtime에서 middleware를 구현하는 방법을 보여줍니다.

## 학습 내용

- Starlette/ASGI 애플리케이션에서 middleware가 작동하는 방식
- BaseHTTPMiddleware를 사용하여 custom middleware 생성
- BedrockAgentCoreApp에 middleware 전달
- 여러 middleware component 연결
- middleware가 활성화된 에이전트를 AgentCore Runtime에 배포

## 사전 요구 사항
- 구성된 AWS credentials
- requirements.txt에서 필수 package 설치
- Amazon Bedrock console에서 모델 액세스 활성화
- 실행 중인 Docker

## 설정

필수 package를 설치하고 dependency를 가져옵니다.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import time

from starlette.middleware.base import BaseHTTPMiddleware

## Middleware 이해

AgentCore Runtime의 middleware는 Starlette ASGI middleware system을 사용합니다. middleware는 위에서 아래 순서로 평가됩니다.

```
Request → Middleware 1 → Middleware 2 → Agent → Middleware 2 → Middleware 1 → Response
```

### BaseHTTPMiddleware Pattern

middleware를 생성하려면 다음 작업을 수행합니다.

1. `BaseHTTPMiddleware` 상속
2. `async def dispatch(self, request, call_next)` 구현
3. 다음 layer를 호출하도록 `response = await call_next(request)` 실행
4. `call_next()` 전에는 request를, 이후에는 response를 처리

`call_next` 함수는 request를 다음 middleware 또는 agent handler에 전달합니다. 호출 전에는 request를 검사하거나 수정하고, 호출 후에는 response를 검사하거나 수정할 수 있습니다.

### 핵심 사항

- Middleware가 애플리케이션을 ASGI component로 감쌈
- 각 middleware가 request와 response를 처리할 수 있음
- Middleware가 목록에 지정된 순서로 실행됨
- middleware와 handler 간 데이터 전달에 `request.state` 사용
- middleware는 stateless로 유지하고 request별 데이터를 instance variable에 저장하지 않음

## 일반적인 Middleware 사용 사례

middleware를 구현하기 전에 일반적인 사용 사례를 살펴봅니다.

### 1. Logging 및 Observability
debugging, auditing, compliance를 위해 모든 request와 response를 추적합니다. timestamp, request 세부 정보, response 상태를 기록하여 production 환경의 에이전트 동작을 파악합니다.

### 2. Metric 수집
request duration, 성공률, 오류 수 같은 성능 metric을 측정합니다. SLA monitoring, bottleneck 식별, 에이전트 성능 최적화에 필요합니다.

### 3. 오류 처리 및 형식 지정
오류 응답을 표준화하고 troubleshooting용 correlation ID를 추가하며 오류를 사용자 친화적 message로 변환합니다. 에이전트 전체에서 일관된 오류 처리를 보장합니다.

### 4. 콘텐츠 Filtering 및 Guardrails
request가 에이전트에 도달하기 전에 Amazon Bedrock Guardrails를 적용하여 filtering합니다. 유해한 콘텐츠를 차단하고 denied topic을 적용하며 policy 위반을 방지합니다. 잘못된 request를 조기에 거부하여 compute 비용을 절감합니다.

### 5. Rate Limiting
user 또는 API key별 request 빈도를 제어하여 남용을 방지하고 공정한 리소스 할당을 보장합니다. 에이전트를 과부하로부터 보호하고 비용을 관리합니다.

### 6. 인증 및 권한 부여
request가 에이전트에 도달하기 전에 API key, JWT token 또는 custom credentials를 검증합니다. agent logic을 복잡하게 만들지 않고 보안을 추가합니다.

이 튜토리얼에서는 가장 일반적인 production pattern을 보여주는 **Logging 및 Metrics**(통합), **오류 처리**, **Guardrails 기반 콘텐츠 Filtering** middleware를 구현합니다.

## 예제 1: Observability Middleware(Logging + Metrics)

이 middleware는 밀접하게 관련된 logging과 metric 수집을 결합하며 다음 작업을 수행합니다.

**Logging:**
- request method, path, timestamp 기록
- response 상태와 duration 기록
- compliance용 audit trail 제공

**Metrics:**
- request 처리 시간 측정
- request 수와 pattern 추적
- 성능 monitoring 지원


**결합하는 이유:** 두 기능 모두 시간을 측정하고 동일한 request/response 데이터에 액세스해야 합니다. 결합하면 overhead를 줄이고 관련 기능을 함께 유지할 수 있습니다.

In [ ]:
# 핵심 개념: async dispatch를 사용하여 request/response 감싸기
class ObservabilityMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        # request 전: log 기록 및 timer 시작
        print(f"REQUEST: {request.method} {request.url.path}")
        start_time = time.time()

        # request 처리
        response = await call_next(request)

        # response 후: duration 기록
        duration = time.time() - start_time
        print(f"RESPONSE: Status {response.status_code} | Duration {duration:.4f}s")

        return response

## 예제 2: 오류 처리 Middleware

이 middleware는 에이전트 전체의 오류 처리를 표준화합니다.

**오류 처리:**
- exception이 에이전트를 중단시키기 전에 catch
- debugging을 위해 전체 context와 함께 오류 기록
- 민감한 오류 세부 정보가 client에 노출되지 않도록 방지

**오류 형식 지정:**
- 일관된 오류 응답 구조 반환
- troubleshooting용 correlation ID 추가
- 사용자 친화적 오류 message 제공

**중요한 이유:** 오류 middleware가 없으면 exception이 내부 세부 정보를 노출하거나 에이전트를 중단시키거나 일관되지 않은 오류 형식을 반환할 수 있습니다. 이 middleware는 graceful degradation과 더 나은 debugging을 보장합니다.

In [ ]:
# 핵심 개념: 오류를 catch하도록 try/except로 감싸기
class ErrorHandlingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        correlation_id = str(uuid.uuid4())

        try:
            response = await call_next(request)
            response.headers["x-correlation-id"] = correlation_id
            return response
        except Exception as e:
            # correlation ID와 함께 오류 기록
            print(f"ERROR: {type(e).__name__}: {str(e)}")
            print(f"Correlation ID: {correlation_id}")

            # 사용자 친화적 오류 반환
            return JSONResponse(
                status_code=500,
                content={
                    "error": "An error occurred",
                    "correlation_id": correlation_id,
                },
            )

## Middleware를 사용하는 Production Agent 생성

Observability 및 오류 처리 middleware가 포함된 agent 파일을 생성합니다.

In [ ]:
%%writefile middleware_agent.py
import time
import json
from datetime import datetime
import traceback
import uuid

from bedrock_agentcore import BedrockAgentCoreApp
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse
from strands import Agent
from strands.models import BedrockModel

# Middleware 1: Observability 설정
class ObservabilityMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        timestamp = datetime.now().isoformat()
        print(f"\n[{timestamp}] REQUEST: {request.method} {request.url.path}")
        start_time = time.time()
        response = await call_next(request)
        duration = time.time() - start_time
        print(f"[{timestamp}] RESPONSE: Status {response.status_code} | Duration {duration:.4f}s")
        response.headers["x-process-time"] = f"{duration:.4f}s"
        return response

# Middleware 2: 오류 처리
class ErrorHandlingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        correlation_id = str(uuid.uuid4())
        try:
            response = await call_next(request)
            response.headers["x-correlation-id"] = correlation_id
            return response
        except Exception as e:
            print(f"\n❌ ERROR: {type(e).__name__}: {str(e)}")
            return JSONResponse(
                status_code=500,
                content={"error": "An error occurred", "correlation_id": correlation_id}
            )

# middleware를 사용하는 app 생성
app = BedrockAgentCoreApp(
    middleware=[
        Middleware(ErrorHandlingMiddleware),
        Middleware(ObservabilityMiddleware),
    ]
)

# 에이전트 초기화
model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")
agent = Agent(model=model, system_prompt="You are a helpful AI assistant.")

@app.entrypoint
def agent_handler(payload, context):
    user_message = payload.get("prompt", "Hello!")
    result = agent(user_message)
    return {"response": result.message}

if __name__ == "__main__":
    app.run()

## AgentCore Runtime에 배포

Observability 및 오류 처리 middleware와 함께 에이전트를 배포합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="middleware_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="middleware_agent",
)

print("✓ Agent configured")

In [ ]:
# 에이전트 실행
launch_result = agentcore_runtime.launch()
print("✓ Agent launched")

In [ ]:
# 배포가 완료될 때까지 대기

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

print(f"\n✓ Deployment complete: {status}")

## 에이전트 테스트

기본 middleware로 에이전트를 테스트합니다.

In [ ]:
import json

response = agentcore_runtime.invoke({"prompt": "What is the capital of France?"})

response_data = json.loads(response["response"][0])
print("Agent Response:")
print("=" * 60)
print(response_data["response"])

print("\n" + "=" * 60)
print("📊 View Middleware Output in CloudWatch Logs:")
print("=" * 60)
print("   /aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>/[runtime-logs]")
print("\nYou'll see:")
print("  - REQUEST: timestamp, method, path")
print("  - RESPONSE: status code, duration")
print("  - Correlation IDs for tracking")

## Guardrail Middleware 추가(선택 사항)

이제 Amazon Bedrock Guardrails를 사용한 콘텐츠 filtering용 세 번째 middleware를 추가합니다.

**사전 요구 사항:**
- AWS 계정에서 Bedrock Guardrails 액세스 활성화
- IAM 권한: `bedrock:CreateGuardrail`, `bedrock:GetGuardrail`

**access denied 오류가 발생하면** 이 섹션을 건너뛸 수 있습니다. 에이전트는 앞에서 배포한 두 middleware로 이미 작동합니다.

In [ ]:
import boto3

bedrock_client = boto3.client("bedrock", region_name="us-east-1")

# guardrail 생성 - name, blockedInputMessaging, blockedOutputsMessaging만 필요
guardrail_response = bedrock_client.create_guardrail(
    name="financial-advice-blocker",
    description="Blocks requests for financial advice",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Financial Advice",
                "definition": "Questions seeking investment advice, stock recommendations, or financial planning guidance",
                "examples": [
                    "Should I invest in cryptocurrency?",
                    "What stocks should I buy?",
                    "How should I plan my retirement savings?",
                ],
                "type": "DENY",
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {
                "type": "PROMPT_ATTACK",
                "inputStrength": "HIGH",
                "outputStrength": "NONE",
            },
        ]
    },
    blockedInputMessaging="This request violates our content policy.",
    blockedOutputsMessaging="This response violates our content policy.",
)

guardrail_id = guardrail_response["guardrailId"]
guardrail_version = guardrail_response["version"]
print(f"✓ Created guardrail: {guardrail_id} (version: {guardrail_version})")

### Guardrail Middleware 개념

이 middleware는 request가 에이전트에 도달하기 전에 검사하고 policy 위반을 차단합니다.

In [ ]:
# 핵심 개념: 에이전트에 전달하기 전에 request 확인
class GuardrailMiddleware(BaseHTTPMiddleware):
    def __init__(self, app, guardrail_id: str):
        super().__init__(app)
        self.guardrail_id = guardrail_id
        self.bedrock_runtime = boto3.client("bedrock-runtime")

    async def dispatch(self, request, call_next):
        if request.method == "POST" and "/invocations" in request.url.path:
            body = await request.body()
            payload = json.loads(body)
            user_prompt = payload.get("prompt", "")

            # guardrail 적용
            result = self.bedrock_runtime.apply_guardrail(
                guardrailIdentifier=self.guardrail_id,
                guardrailVersion="DRAFT",
                source="INPUT",
                content=[{"text": {"text": user_prompt}}],
            )

            # guardrail이 개입하면 차단하고 agent 호환 응답 반환
            if result["action"] == "GUARDRAIL_INTERVENED":
                blocked_msg = result["outputs"][0]["text"] if result.get("outputs") else "Content blocked by guardrail"
                print(f"🛡️ Guardrail blocked: {user_prompt[:50]}...")

                # agent 응답 형식으로 반환
                return JSONResponse(status_code=200, content={"response": blocked_msg})

        return await call_next(request)

### Guardrail Middleware로 에이전트 업데이트

기존 에이전트에 guardrail middleware를 추가합니다. `auto_update_on_conflict=True`를 사용하여 에이전트를 제자리에서 업데이트합니다.

In [ ]:
%%writefile middleware_agent.py
import time
import json
from datetime import datetime
import traceback
import uuid
import os

import boto3
from bedrock_agentcore import BedrockAgentCoreApp
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse
from strands import Agent
from strands.models import BedrockModel

# Middleware 1: Observability 설정
class ObservabilityMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        timestamp = datetime.now().isoformat()
        print(f"\n[{timestamp}] REQUEST: {request.method} {request.url.path}")
        start_time = time.time()
        response = await call_next(request)
        duration = time.time() - start_time
        print(f"[{timestamp}] RESPONSE: Status {response.status_code} | Duration {duration:.4f}s")
        response.headers["x-process-time"] = f"{duration:.4f}s"
        return response

# Middleware 2: 오류 처리
class ErrorHandlingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        correlation_id = str(uuid.uuid4())
        try:
            response = await call_next(request)
            response.headers["x-correlation-id"] = correlation_id
            return response
        except Exception as e:
            print(f"\n❌ ERROR: {type(e).__name__}: {str(e)}")
            return JSONResponse(
                status_code=500,
                content={"error": "An error occurred", "correlation_id": correlation_id}
            )

# Middleware 3: Guardrail(신규)
class GuardrailMiddleware(BaseHTTPMiddleware):
    def __init__(self, app, guardrail_id: str, guardrail_version: str = 'DRAFT'):
        super().__init__(app)
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version
        self.bedrock_runtime = boto3.client('bedrock-runtime')
    
    async def dispatch(self, request, call_next):
        if request.method == 'POST' and '/invocations' in request.url.path:
            body = await request.body()
            try:
                payload = json.loads(body)
                user_prompt = payload.get('prompt', '')
                
                result = self.bedrock_runtime.apply_guardrail(
                    guardrailIdentifier=self.guardrail_id,
                    guardrailVersion=self.guardrail_version,
                    source='INPUT',
                    content=[{'text': {'text': user_prompt}}]
                )
                
                if result['action'] == 'GUARDRAIL_INTERVENED':
                    blocked_msg = result['outputs'][0]['text'] if result.get('outputs') else 'Content blocked'
                    print(f"🛡️ Guardrail blocked: {user_prompt[:50]}...")
                    return JSONResponse(status_code=200, content={'response': blocked_msg})
                
                print(f"✓ Guardrail passed: {user_prompt[:50]}...")
            except Exception as e:
                print(f"⚠️ Guardrail check failed: {e}")
        
        return await call_next(request)

# 환경에서 guardrail ID 가져오기
GUARDRAIL_ID = os.environ.get('GUARDRAIL_ID', '')

# middleware 목록 구성
middleware_list = [
    Middleware(ErrorHandlingMiddleware),
    Middleware(ObservabilityMiddleware),
]

if GUARDRAIL_ID:
    middleware_list.insert(0, Middleware(GuardrailMiddleware, guardrail_id=GUARDRAIL_ID))
    print(f"✓ Guardrail enabled: {GUARDRAIL_ID}")

app = BedrockAgentCoreApp(middleware=middleware_list)

# 에이전트 초기화
model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")
agent = Agent(model=model, system_prompt="You are a helpful AI assistant.")

@app.entrypoint
def agent_handler(payload, context):
    user_message = payload.get("prompt", "Hello!")
    result = agent(user_message)
    return {"response": result.message}

if __name__ == "__main__":
    app.run()

In [ ]:
# 기존 에이전트를 guardrail로 업데이트
launch_result = agentcore_runtime.launch(env_vars={"GUARDRAIL_ID": guardrail_id}, auto_update_on_conflict=True)

# 업데이트 대기
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

print(f"\n✓ Agent updated with guardrail: {status}")

### Guardrail Middleware 테스트

금융 조언 질문(차단)과 일반 질문(허용)으로 테스트합니다.

In [ ]:
# 테스트 1: 금융 조언(차단되어야 함)
print("Test 1: Financial Advice Question (should be blocked)")
print("=" * 60)

response = agentcore_runtime.invoke({"prompt": "Should I invest in cryptocurrency? What stocks should I buy?"})
response_data = json.loads(response["response"][0])
print(f"Response: {response_data['response']}")

print("\n" + "=" * 60)
print("\nTest 2: Normal Question (should pass)")
print("=" * 60)

response = agentcore_runtime.invoke({"prompt": "What is the capital of France?"})
response_data = json.loads(response["response"][0])
print(f"Response: {response_data['response']}")

print("\n" + "=" * 60)
print("📊 Check CloudWatch Logs to see:")
print("  - Guardrail blocking financial advice")
print("  - Guardrail passing normal questions")
print("  - Request timing and correlation IDs")

## 리소스 정리

작업을 마치면 배포된 리소스를 정리합니다.

In [ ]:
agentcore_runtime.destroy()

## 요약

다음 내용을 학습했습니다.

✓ Starlette ASGI system을 사용하여 AgentCore Runtime에서 middleware가 작동하는 방식  
✓ BaseHTTPMiddleware로 custom middleware 생성  
✓ 일반적인 pattern: logging, metric, 오류 처리, 콘텐츠 filtering  
✓ request filtering을 위한 Amazon Bedrock Guardrails 통합  
✓ 여러 middleware component 연결  
✓ middleware가 활성화된 에이전트를 AgentCore Runtime에 배포하고 테스트  

### 핵심 요점

- Middleware가 cross-cutting concern을 명확하게 분리
- Middleware가 순서대로 실행되며 목록의 첫 번째 항목이 나머지를 모두 감쌈
- `call_next()`를 사용하여 다음 layer로 제어 전달
- Guardrails가 agent 처리 전에 request를 조기에 차단 가능
- 동시 request 처리를 위해 middleware를 stateless로 유지
- 배포된 에이전트의 middleware log가 CloudWatch Logs에 표시됨